# Real Market Data and De-Time Feature Factory

This notebook downloads real close prices, audits them, and computes walk-forward De-Time features. It intentionally stops if real market data is not available.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in [ROOT / "src", ROOT / "examples"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from quant_trading.data import fetch_yahoo_prices, fetch_yahoo_ohlcv, data_audit_report, DEFAULT_UNIVERSES
from quant_trading.features import decompose_one_series, walkforward_decompose, build_feature_table
from quant_trading.signals import (
    trend_pullback_signals,
    residual_mean_reversion_signals,
    turtle_donchian_signals,
    pair_trading_weights,
    cross_sectional_rotation_weights,
    residual_stress_filter,
)
from quant_trading.backtest import backtest_weights, backtest_long_short_signals, summarize_returns

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "005930.KS", "000660.KS", "BTC-USD", "ETH-USD"]
prices = fetch_yahoo_prices(
    tickers,
    start="2018-01-01",
    interval="1d",
    cache_dir=ROOT / "examples" / "quant_trading" / "data" / "cache",
    min_observations=500,
)
prices.tail()

In [ ]:
audit = data_audit_report(prices)
audit

## Decompose one asset first

The output frame includes the original price, transformed price, trend, cycle/seasonal component, residual, and standardized features.

In [ ]:
frame = decompose_one_series(prices["AAPL"], method="STL", period=63)
frame[["price", "trend", "season", "residual", "trend_slope", "residual_z"]].tail()

In [ ]:
ax = frame[["transformed_price", "trend"]].plot(figsize=(10, 4), title="AAPL log price and De-Time trend")
ax.set_xlabel("date")
plt.show()

## Walk-forward feature factory

This step recomputes De-Time on rolling historical windows and keeps only the last row of each window. This avoids full-sample decomposition leakage.

In [ ]:
features = walkforward_decompose(
    prices[["AAPL", "MSFT", "NVDA"]],
    method="STL",
    period=63,
    train_window=252,
    step=21,
)
feature_table = build_feature_table(prices[["AAPL", "MSFT", "NVDA"]], features)
feature_table.tail()